# Nassau Candy — Exploratory Data AnalysisThis notebook documents the manual data audit that informed every downstreammodeling decision in this project — most importantly, the discovery thatthe `Ship Date` column is corrupted and cannot be used for real lead-timecalculation.Run the cells top to bottom. No external data or network access required —everything reads from `../data/Nassau_Candy_Distributor.csv`.

In [ ]:
import syssys.path.insert(0, "../src")import pandas as pdimport numpy as nppd.set_option("display.max_columns", None)df = pd.read_csv("../data/Nassau_Candy_Distributor.csv")df.shape

## 1. First look

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

## 2. The Ship Date anomaly (critical finding)Order Date and Ship Date both look like normal dates at first glance, butparsing them and computing the gap reveals a serious problem.

In [ ]:
df["OrderDate_p"] = pd.to_datetime(df["Order Date"], format="%d-%m-%Y")df["ShipDate_p"] = pd.to_datetime(df["Ship Date"], format="%d-%m-%Y")print("Order date range:", df["OrderDate_p"].min(), "to", df["OrderDate_p"].max())print("Ship date range:", df["ShipDate_p"].min(), "to", df["ShipDate_p"].max())df["raw_lead_days"] = (df["ShipDate_p"] - df["OrderDate_p"]).dt.daysdf["raw_lead_days"].describe()

**Verdict:** lead times of 900–1600+ days are not physically plausible forcandy distribution. The year of Ship Date clusters at fixed multi-year offsetsfrom Order Date rather than a realistic few-day gap — confirmed below.

In [ ]:
df["order_year"] = df["OrderDate_p"].dt.yeardf["ship_year"] = df["ShipDate_p"].dt.yearpd.crosstab(df["order_year"], df["ship_year"])

**Decision:** `Ship Date` is excluded from modeling. The project insteadsimulates a realistic lead time driven by the real `Ship Mode` column —see `src/feature_engineering.py::add_simulated_lead_time`. This is clearlylabeled throughout the app.

## 3. Financial data sanity checks

In [ ]:
diff = (df["Cost"] + df["Gross Profit"] - df["Sales"]).abs()print("Rows where Cost + Profit != Sales:", (diff > 0.01).sum())print("Negative Sales:", (df["Sales"] <= 0).sum())print("Negative Units:", (df["Units"] <= 0).sum())print("Negative Cost:", (df["Cost"] < 0).sum())

All financial figures are internally consistent — Sales always equalsCost + Gross Profit exactly, and there are no negative values. This data isused as-is with no transformation.

## 4. Categorical distributions

In [ ]:
for col in ["Division", "Region", "Ship Mode", "Country/Region"]:    print(col, ":", sorted(df[col].unique()))

In [ ]:
df["Division"].value_counts()

**Note:** there is no factory table in this dataset. `Division` is theonly real production-grouping signal, which is why this project derives a3-factory network from it (see `src/factory_mapping.py` for full disclosure).

## 5. Outlier scan (IQR method)

In [ ]:
for col in ["Sales", "Units", "Gross Profit", "Cost"]:    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)    iqr = q3 - q1    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr    n_out = ((df[col] < lo) | (df[col] > hi)).sum()    print(f"{col}: {n_out} outliers (bounds {lo:.2f} to {hi:.2f})")

These outliers are legitimate large/small orders (e.g. bulk purchases),not data errors, and are retained in modeling.

## 6. Correlations among numeric features

In [ ]:
numeric_cols = ["Sales", "Units", "Gross Profit", "Cost"]df[numeric_cols].corr()

## 7. Geographic coverage616 unique City/State/Country combinations roll up to a manageable 59state/province values — small enough for a reliable public centroid lookup(see `src/geo_lookup.py`), but too many for individual city-level geocodingwithout an external API.

In [ ]:
print("Unique city/state/country combos:", df[["City","State/Province","Country/Region"]].drop_duplicates().shape[0])print("Unique states/provinces:", df["State/Province"].nunique())

## SummaryThis audit directly informs every design decision documented in`reports/data_quality_report.json` and surfaced live in the app's**Data Quality Report** page. The full production pipeline (cleaning,feature engineering, modeling, clustering, optimization) lives in `src/`and is orchestrated by `src/run_pipeline.py`.